In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [3]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## Preprocessing the data
### Dropping unnecessary columns

cleaned_data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
cleaned_data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
label_encoder_gender = LabelEncoder()
cleaned_data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
cleaned_data.head(10)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
5,645,Spain,1,44,8,113755.78,2,1,0,149756.71,1
6,822,France,1,50,7,0.00,2,1,1,10062.80,0
7,376,Germany,0,29,4,115046.74,4,1,0,119346.88,1
8,501,France,1,44,4,142051.07,2,0,1,74940.50,0
9,684,France,1,27,2,134603.88,1,1,1,71725.73,0


In [7]:
# one hot encoding for geography column
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geography = OneHotEncoder()

geography_encoded = onehot_encoder_geography.fit_transform(cleaned_data[['Geography']])
geography_encoded

geo_encoded_df = pd.DataFrame(geography_encoded.toarray(),columns=onehot_encoder_geography.get_feature_names_out(['Geography']))
geo_encoded_df.head(10)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
5,0.0,0.0,1.0
6,1.0,0.0,0.0
7,0.0,1.0,0.0
8,1.0,0.0,0.0
9,1.0,0.0,0.0


In [9]:
cleaned_data = pd.concat([cleaned_data.drop('Geography',axis=1),geo_encoded_df],axis=1)
cleaned_data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
x = cleaned_data.drop('EstimatedSalary',axis=1)
y = cleaned_data['EstimatedSalary']

In [12]:
X_train,X_test,Y_train,Y_test = train_test_split(x,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [13]:
## Save the encoders and scaler
with open('label_encoder_gender_regression.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo_regression.pkl','wb') as file:
    pickle.dump(onehot_encoder_geography,file)

with open('scaler_regression.pkl','wb') as file:
    pickle.dump(scaler,file)

##### ANN Regression statement

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [15]:
## Build our ANN Model
regression_model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1)    #output layer for regression
])

c:\Study\Gen AI Prep\ANN Classification\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
## Compile the model
regression_model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
regression_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
## Set up the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir = "regression_logs/fit/"
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [23]:
# Set up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [24]:
history = regression_model.fit(
    X_train,Y_train,
    validation_data = (X_test,Y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensorflow_callback]
)

Epoch 1/100


250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 100377.2266 - mae: 100377.2266 - val_loss: 98518.1328 - val_mae: 98518.1328
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 99665.0469 - mae: 99665.0469 - val_loss: 97094.0000 - val_mae: 97094.0000
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 97202.4531 - mae: 97202.4531 - val_loss: 93480.6250 - val_mae: 93480.6250
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 92363.4219 - mae: 92363.4219 - val_loss: 87435.3984 - val_mae: 87435.3984
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 85241.7422 - mae: 85241.7422 - val_loss: 79509.5703 - val_mae: 79509.5703
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 76686.9062 - mae: 76686.9062 - val_loss: 71018.0312 - val_mae: 71018.0312
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 68125.8281 - mae: 68125.8281 - val_loss: 63185.8906 - val_mae: 63185.8906
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 6082

In [26]:
%load_ext tensorboard

In [30]:
%tensorboard --logdir regression_logs/fit

Reusing TensorBoard on port 6006 (pid 12364), started 0:06:04 ago. (Use '!kill 12364' to kill it.)

In [32]:
## Evaluate the model
test_loss,test_mae = regression_model.evaluate(X_test,Y_test)
print(test_loss)
print(test_mae)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 50285.2383 - mae: 50285.2383
50285.23828125
50285.23828125


In [33]:
regression_model.save('salary_regression_model.h5')